In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

1st try with sequence 


In [6]:
import os
import shutil
from sklearn.model_selection import train_test_split

base_path = '/kaggle/input/et-cropped-sharpened-dataset'
output_path = '/kaggle/working/split_dataset'

severity_groups = ['low', 'mild', 'medium', 'high']

subject_map = {cat: set() for cat in severity_groups}

# 1. Map Subject IDs
for cat in severity_groups:
    cat_dir = os.path.join(base_path, cat)
    if not os.path.exists(cat_dir):
        continue
    files = os.listdir(cat_dir)
    for f in files:
        sub_id = f.split('-')[0]
        subject_map[cat].add(sub_id)

# 2. Split Subjects using train_test_split
train_subjects = []
val_subjects = []

for cat, subjects in subject_map.items():
    subs = sorted(list(subjects))
    # This ensures a 20% split regardless of how many subjects are in the folder
    tr, vl = train_test_split(subs, test_size=0.2, random_state=42)
    train_subjects.extend(tr)
    val_subjects.extend(vl)

# 3. Copy files
for cat in severity_groups:
    cat_path = os.path.join(base_path, cat)
    if not os.path.exists(cat_path):
        continue
    for f in os.listdir(cat_path):
        sub_id = f.split('-')[0]
        split = 'train' if sub_id in train_subjects else 'val'
        
        target_dir = os.path.join(output_path, split, cat)
        os.makedirs(target_dir, exist_ok=True)
        shutil.copy(os.path.join(cat_path, f), os.path.join(target_dir, f))

print(f"Split complete. Train: {len(train_subjects)} subs, Val: {len(val_subjects)} subs.")

Split complete. Train: 32 subs, Val: 8 subs.


In [11]:
import tensorflow as tf
from tensorflow.keras import layers, models
import os

from tensorflow.keras import mixed_precision
policy = mixed_precision.Policy('mixed_float16')
mixed_precision.set_global_policy(policy)
# Note: Your model.summary() will now show 'mixed_float16' for most layers.

class SpatialSeeker(layers.Layer):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)

    def build(self, input_shape):
        # input_shape will be (batch, num_patches, channels)
        # We create the weights here once we know the 'channels' count
        self.conv = layers.Conv1D(1, kernel_size=7, padding='same', activation='sigmoid')
        super().build(input_shape)

    def call(self, x):
        # x shape: (batch, patches, channels)
        avg_p = tf.reduce_mean(x, axis=-1, keepdims=True)
        max_p = tf.reduce_max(x, axis=-1, keepdims=True)
        # Find the biomarker focus
        attn = self.conv(tf.concat([avg_p, max_p], axis=-1))
        return x * attn

    def compute_output_shape(self, input_shape):
        # This tells TimeDistributed that the shape DOES NOT change
        return input_shape

def build_temporal_hd_transformer(sequence_length=10, input_shape=(300, 300, 3), num_classes=4):
    inputs = layers.Input(shape=(sequence_length,) + input_shape)
    
    # --- ADDED: Gaussian Noise to prevent person-memorization ---
    x = layers.TimeDistributed(layers.GaussianNoise(0.1))(inputs) 
    
    # Surgical Augmentation
    data_augmentation = tf.keras.Sequential([
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.05),
        layers.RandomTranslation(0.05, 0.05),
        layers.RandomContrast(0.1)
    ])
    # --- ADDED: Applying augmentation to the noisy input ---
    x = layers.TimeDistributed(data_augmentation)(x)

    # Backbone
    base_model = tf.keras.applications.EfficientNetV2B0(
        input_shape=input_shape, include_top=False, weights='imagenet'
    )
    
    # Initial State: Backbone is fully trainable but we will control 
    # the unfreezing in the training stages.
    base_model.trainable = True

    x = layers.TimeDistributed(base_model)(x)
    
    # Reshape for Attention
    f_h, f_w, channels = x.shape[2], x.shape[3], x.shape[4]
    num_patches = f_h * f_w
    x = layers.Reshape((sequence_length, num_patches, channels))(x)
    
    # FIXED: Use the class-based seeker
    x = layers.TimeDistributed(SpatialSeeker())(x)
    
    # Temporal Processing
    x = layers.TimeDistributed(layers.GlobalMaxPooling1D())(x)

    # Transformer Block
    res = x
    x = layers.LayerNormalization(epsilon=1e-6)(x)
    # --- INCREASED DROPOUT: From 0.3 to 0.5 to stop overfitting ---
    x = layers.MultiHeadAttention(num_heads=8, key_dim=128, dropout=0.5)(x, x) 
    x = layers.Add()([res, x])

    # FFN Reasoning
    res = x
    x = layers.LayerNormalization(epsilon=1e-6)(x)
    x = layers.Dense(channels * 2, activation=tf.nn.gelu)(x)
    x = layers.Dense(channels)(x)
    x = layers.Add()([res, x])

    # Classification
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    model = models.Model(inputs, outputs)
    
    # REMOVED ExponentialDecay logic here to prevent callback conflicts
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4), 
        loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.05),
        metrics=['accuracy']
    )
    return model

In [12]:
import numpy as np
import cv2

class ETSequenceGenerator(tf.keras.utils.PyDataset): # Updated to PyDataset
    def __init__(self, directory, seq_len=10, batch_size=16, target_size=(300, 300), **kwargs):
        super().__init__(**kwargs) # Fixes the UserWarning
        self.directory = directory
        self.seq_len = seq_len
        self.batch_size = batch_size
        self.target_size = target_size
        self.classes = ['low', 'mild', 'medium', 'high']
        self.data = self._build_sequences()

    def _build_sequences(self):
        sequences = []
        for label, cls in enumerate(self.classes):
            path = os.path.join(self.directory, cls)
            if not os.path.exists(path): continue
            files = sorted(os.listdir(path))
            
            # Group by Subject AND Eye-side for clean motion paths
            subject_dict = {}
            for f in files:
                parts = f.split('-')
                sub_id = parts[0]
                eye_side = parts[-1].replace('.jpg', '')
                group_key = f"{sub_id}_{eye_side}"
                
                if group_key not in subject_dict: subject_dict[group_key] = []
                subject_dict[group_key].append(os.path.join(path, f))
            
            for group_key, frames in subject_dict.items():
                for i in range(0, len(frames) - self.seq_len + 1, self.seq_len):
                    sequences.append((frames[i:i+self.seq_len], label))
        return sequences

    def __len__(self):
        return int(np.floor(len(self.data) / self.batch_size))

    def __getitem__(self, index):
        batch_data = self.data[index * self.batch_size:(index + 1) * self.batch_size]
        X = np.zeros((self.batch_size, self.seq_len, *self.target_size, 3))
        y = np.zeros((self.batch_size, 4))
        
        for i, (frames, label) in enumerate(batch_data):
            for j, frame_path in enumerate(frames):
                img = cv2.imread(frame_path)
                img = cv2.resize(cv2.cvtColor(img, cv2.COLOR_BGR2RGB), self.target_size)
                X[i, j] = img / 255.0
            y[i, label] = 1
        return X, y

In [13]:
# 1. Initialize the generator pointing to your split training data
verify_gen = ETSequenceGenerator(
    directory='/kaggle/working/split_dataset/train', 
    seq_len=5, 
    batch_size=2 # Small batch just to check
)

# 2. Extract the first batch
X_batch, y_batch = verify_gen[0]

print(f"--- Verification Report ---")
print(f"Batch Shape: {X_batch.shape} (Batch size, Sequence, Height, Width, Channels)")
print(f"Labels Shape: {y_batch.shape}\n")

# 3. Manually check the logic inside the generator's data list
for i in range(3): # Check first 3 sequences
    frames, label = verify_gen.data[i]
    print(f"Sequence {i} (Category Index: {label}):")
    for frame_path in frames:
        # Print just the filename to check Subject ID and Frame Number
        print(f"  - {os.path.basename(frame_path)}")
    print("-" * 30)

--- Verification Report ---
Batch Shape: (2, 5, 300, 300, 3) (Batch size, Sequence, Height, Width, Channels)
Labels Shape: (2, 4)

Sequence 0 (Category Index: 0):
  - 12-100-low-le.jpg
  - 12-110-low-le.jpg
  - 12-120-low-le.jpg
  - 12-130-low-le.jpg
  - 12-140-low-le.jpg
------------------------------
Sequence 1 (Category Index: 0):
  - 12-150-low-le.jpg
  - 12-160-low-le.jpg
  - 12-170-low-le.jpg
  - 12-180-low-le.jpg
  - 12-190-low-le.jpg
------------------------------
Sequence 2 (Category Index: 0):
  - 12-200-low-le.jpg
  - 12-210-low-le.jpg
  - 12-220-low-le.jpg
  - 12-230-low-le.jpg
  - 12-240-low-le.jpg
------------------------------


In [14]:
# Create model
model = build_temporal_hd_transformer()

# Prepare generators
train_gen = ETSequenceGenerator('/kaggle/working/split_dataset/train')
val_gen = ETSequenceGenerator('/kaggle/working/split_dataset/val')

callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=30, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=10, min_lr=1e-6)
]

# The final push
history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=150,
    class_weight={0: 1.2, 1: 1.3, 2: 1.4, 3: 1.2},
    callbacks=callbacks
)

Epoch 1/150


E0000 00:00:1769196096.897105      55 meta_optimizer.cc:967] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape inStatefulPartitionedCall/functional_7_1/time_distributed_13_1/block2b_drop_19/stateless_dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer


ResourceExhaustedError: Graph execution error:

Detected at node functional_7_1/time_distributed_13_1/block6a_expand_bn_19/moments/SquaredDifference defined at (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main

  File "<frozen runpy>", line 88, in _run_code

  File "/usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py", line 37, in <module>

  File "/usr/local/lib/python3.12/dist-packages/traitlets/config/application.py", line 992, in launch_instance

  File "/usr/local/lib/python3.12/dist-packages/ipykernel/kernelapp.py", line 712, in start

  File "/usr/local/lib/python3.12/dist-packages/tornado/platform/asyncio.py", line 211, in start

  File "/usr/lib/python3.12/asyncio/base_events.py", line 645, in run_forever

  File "/usr/lib/python3.12/asyncio/base_events.py", line 1999, in _run_once

  File "/usr/lib/python3.12/asyncio/events.py", line 88, in _run

  File "/usr/local/lib/python3.12/dist-packages/ipykernel/kernelbase.py", line 510, i

In [17]:
# Create model
model = build_temporal_hd_transformer()

# Prepare generators
train_gen = ETSequenceGenerator('/kaggle/working/split_dataset/train')
val_gen = ETSequenceGenerator('/kaggle/working/split_dataset/val')

callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=30, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=10, min_lr=1e-6)
]

# --- STAGE 1: Lock Backbone, Train Top ---
print("--- Starting Stage 1: Settle Transformer ---")
backbone = model.layers[3]
backbone.trainable = False 

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4), 
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.05),
    metrics=['accuracy']
)

# Use the correct variable name: 'callbacks'
history1 = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=30, 
    class_weight={0: 1.5, 1: 1.0, 2: 1.2, 3: 1.0}, 
    callbacks=callbacks 
)

# --- STAGE 2: Deep Fine-Tuning (SMART ACCESS) ---
print("\n--- Starting Stage 2: Deep Fine-Tuning ---")

# Automatically find the backbone wrapper by its type or name
backbone_wrapper = None
for layer in model.layers:
    if 'time_distributed' in layer.name and hasattr(layer, 'layer'):
        if 'efficientnet' in layer.layer.name:
            backbone_wrapper = layer
            break

if backbone_wrapper is None:
    raise ValueError("Could not find the EfficientNet backbone in the model!")

backbone_model = backbone_wrapper.layer
print(f"Success: Found {backbone_model.name}")

# Unfreeze and protect early layers
backbone_model.trainable = True
for layer in backbone_model.layers[:-40]: 
    layer.trainable = False

# Re-compile to stabilize the gradients
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-6),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=['accuracy']
)

history2 = model.fit(
    train_gen,
    validation_data=val_gen,
    initial_epoch=30, # Match Stage 1 end
    epochs=150,
    class_weight={0: 1.5, 1: 1.0, 2: 1.2, 3: 1.0},
    callbacks=callbacks
)


--- Starting Stage 1: Settle Transformer ---
Epoch 1/30
19/19 ━━━━━━━━━━━━━━━━━━━━ 171s 3s/step - accuracy: 0.1659 - loss: 9.9873 - val_accuracy: 0.2969 - val_loss: 11.3708 - learning_rate: 1.0000e-04
Epoch 2/30
19/19 ━━━━━━━━━━━━━━━━━━━━ 17s 867ms/step - accuracy: 0.2833 - loss: 14.1589 - val_accuracy: 0.0781 - val_loss: 14.7204 - learning_rate: 1.0000e-04
Epoch 3/30
19/19 ━━━━━━━━━━━━━━━━━━━━ 17s 861ms/step - accuracy: 0.0920 - loss: 17.3590 - val_accuracy: 0.0781 - val_loss: 14.7204 - learning_rate: 1.0000e-04
Epoch 4/30
19/19 ━━━━━━━━━━━━━━━━━━━━ 17s 863ms/step - accuracy: 0.1533 - loss: 16.6169 - val_accuracy: 0.0781 - val_loss: 14.7204 - learning_rate: 1.0000e-04
Epoch 5/30
19/19 ━━━━━━━━━━━━━━━━━━━━ 17s 859ms/step - accuracy: 0.3078 - loss: 13.4913 - val_accuracy: 0.0781 - val_loss: 14.7204 - learning_rate: 1.0000e-04
Epoch 6/30
19/19 ━━━━━━━━━━━━━━━━━━━━ 17s 857ms/step - accuracy: 0.2220 - loss: 16.8428 - val_accuracy: 0.0781 - val_loss: 14.7204 - learning_rate: 1.0000e-04
Epoc

KeyboardInterrupt: 

2nd try with sequence 

In [8]:
import os
import shutil
import numpy as np
from sklearn.model_selection import StratifiedGroupKFold

base_path = '/kaggle/input/et-cropped-sharpened-dataset'
output_path = '/kaggle/working/split_dataset'
severity_groups = ['low', 'mild', 'medium', 'high']

# 1. Map Subject IDs and labels
all_subjects = []
all_labels = []
subject_to_files = {}

for label_idx, cat in enumerate(severity_groups):
    cat_dir = os.path.join(base_path, cat)
    if not os.path.exists(cat_dir):
        continue
    files = os.listdir(cat_dir)
    for f in files:
        sub_id = f.split('-')[0]
        if sub_id not in subject_to_files:
            subject_to_files[sub_id] = []
            all_subjects.append(sub_id)
            all_labels.append(label_idx)
        subject_to_files[sub_id].append((cat, f))

# 2. Perform Stratified Group Split
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
train_idx, val_idx = next(sgkf.split(all_subjects, all_labels, groups=all_subjects))

train_subjects = [all_subjects[i] for i in train_idx]
val_subjects = [all_subjects[i] for i in val_idx]

# 3. Compute Class Weights
class_counts = {cls: 0 for cls in severity_groups}
for sub_id in train_subjects:
    for cat, f in subject_to_files.get(sub_id, []):
        class_counts[cat] += 1

total_samples = sum(class_counts.values())
class_weights = {i: total_samples/(4*count) if count > 0 else 1.0 
                 for i, (cls, count) in enumerate(class_counts.items())}

# 4. Copy Files to working directory
for sub_id in train_subjects:
    for cat, f in subject_to_files.get(sub_id, []):
        target_dir = os.path.join(output_path, 'train', cat)
        os.makedirs(target_dir, exist_ok=True)
        shutil.copy(os.path.join(base_path, cat, f), os.path.join(target_dir, f))

for sub_id in val_subjects:
    for cat, f in subject_to_files.get(sub_id, []):
        target_dir = os.path.join(output_path, 'val', cat)
        os.makedirs(target_dir, exist_ok=True)
        shutil.copy(os.path.join(base_path, cat, f), os.path.join(target_dir, f))

print(f"Split complete. Train: {len(train_subjects)} subs, Val: {len(val_subjects)} subs.")
print(f"Class weights: {class_weights}")

Split complete. Train: 32 subs, Val: 8 subs.
Class weights: {0: 0.8888888888888888, 1: 1.0, 2: 1.0, 3: 1.1428571428571428}


In [12]:
import tensorflow as tf
from tensorflow.keras import layers, models

class TemporalPositionalEncoding(layers.Layer):
    def __init__(self, max_seq_len, projection_dim, **kwargs):
        super().__init__(**kwargs)
        self.max_seq_len = max_seq_len
        self.projection_dim = projection_dim
        
    def build(self, input_shape):
        positions = tf.range(self.max_seq_len, dtype=tf.float32)[:, tf.newaxis]
        div_term = tf.exp(tf.range(0, self.projection_dim, 2, dtype=tf.float32) * -(tf.math.log(10000.0) / self.projection_dim))
        angles = positions * div_term
        pos_enc = tf.concat([tf.sin(angles), tf.cos(angles)], axis=-1)
        self.pos_encoding = tf.Variable(pos_enc, trainable=False, name='temporal_pos_encoding')
        super().build(input_shape)
    
    def call(self, x):
        seq_len = tf.shape(x)[1]
        return x + self.pos_encoding[:seq_len, :]

class SpatialSeeker(layers.Layer):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
    
    def build(self, input_shape):
        # input_shape is (batch, patches, channels)
        self.conv = layers.Conv1D(1, kernel_size=7, padding='same', activation='sigmoid')
        super().build(input_shape)
    
    def call(self, x):
        avg_p = tf.reduce_mean(x, axis=-1, keepdims=True)
        max_p = tf.reduce_max(x, axis=-1, keepdims=True)
        attn = self.conv(tf.concat([avg_p, max_p], axis=-1))
        return x * attn

    # [FIXED] Added this method to satisfy TimeDistributed requirement
    def compute_output_shape(self, input_shape):
        return input_shape

class SeverityQueryAttention(layers.Layer):
    def __init__(self, key_dim, **kwargs):
        super().__init__(**kwargs)
        self.key_dim = key_dim
    def build(self, input_shape):
        self.query_token = self.add_weight(shape=(1, 1, input_shape[-1]),
                                          initializer='random_normal', trainable=True)
        self.mha = layers.MultiHeadAttention(num_heads=1, key_dim=self.key_dim)
        super().build(input_shape)
    def call(self, x):
        batch_size = tf.shape(x)[0]
        query = tf.tile(self.query_token, [batch_size, 1, 1])
        return self.mha(query, x, x)

def build_temporal_hd_transformer(sequence_length=10, input_shape=(300, 300, 3), num_classes=4):
    inputs = layers.Input(shape=(None,) + input_shape)
    x = layers.TimeDistributed(layers.GaussianNoise(0.1))(inputs)
    
    data_aug = tf.keras.Sequential([
        layers.RandomFlip("horizontal"), layers.RandomRotation(0.05),
        layers.RandomTranslation(0.05, 0.05), layers.RandomContrast(0.1)
    ])
    x = layers.TimeDistributed(data_aug)(x)

    base_model = tf.keras.applications.EfficientNetV2B0(input_shape=input_shape, include_top=False, weights='imagenet')
    x = layers.TimeDistributed(base_model)(x)
    
    f_h, f_w, channels = x.shape[2], x.shape[3], x.shape[4]
    x = layers.Reshape((-1, f_h * f_w, channels))(x)
    
    # [FIXED] Updated noise_shape to match the 4D tensor (batch, seq, patches, channels)
    x = layers.Dropout(0.2, noise_shape=(None, None, 1, 1))(x)
    
    x = layers.TimeDistributed(SpatialSeeker())(x)
    x = layers.TimeDistributed(layers.GlobalMaxPooling1D())(x)
    x = TemporalPositionalEncoding(sequence_length, channels)(x)
    
    for _ in range(3):
        res = x
        x = layers.LayerNormalization(epsilon=1e-6)(x)
        attn = layers.MultiHeadAttention(num_heads=8, key_dim=128, dropout=0.5)(x, x)
        x = layers.Add()([res, attn])
        
        res_ffn = x
        x = layers.LayerNormalization(epsilon=1e-6)(x)
        x = layers.Dense(channels * 2, activation=tf.nn.gelu)(x)
        x = layers.Dense(channels)(x)
        x = layers.Add()([res_ffn, x])
    
    x = SeverityQueryAttention(key_dim=128)(x)
    x = layers.Flatten()(x)
    outputs = layers.Dense(num_classes, activation='softmax')(layers.Dropout(0.5)(x))
    return models.Model(inputs, outputs)

In [10]:
import cv2
import numpy as np

class ETSequenceGenerator(tf.keras.utils.PyDataset):
    def __init__(self, directory, seq_len=10, batch_size=16, target_size=(300, 300), subject_aware=False, **kwargs):
        super().__init__(**kwargs)
        self.directory, self.seq_len, self.batch_size, self.target_size = directory, seq_len, batch_size, target_size
        self.classes = ['low', 'mild', 'medium', 'high']
        self.subject_aware = subject_aware
        self.data = self._build_sequences()

    def _build_sequences(self):
        sequences = []
        for label, cls in enumerate(self.classes):
            path = os.path.join(self.directory, cls)
            if not os.path.exists(path): continue
            files = sorted(os.listdir(path))
            subj_dict = {}
            for f in files:
                key = f"{f.split('-')[0]}_{f.split('-')[-1].replace('.jpg', '')}"
                if key not in subj_dict: subj_dict[key] = []
                subj_dict[key].append(os.path.join(path, f))
            
            for key, frames in subj_dict.items():
                for i in range(0, len(frames) - self.seq_len + 1, 1):
                    if np.random.random() > 0.8 and i > 0: continue
                    sequences.append((frames[i:i+self.seq_len], label, key))
        
        if self.subject_aware:
            groups = {}
            for s, l, k in sequences:
                if k not in groups: groups[k] = []
                groups[k].append((s, l))
            return groups
        return sequences

    def __len__(self):
        return len(self.data) if self.subject_aware else int(np.floor(len(self.data)/self.batch_size))

    def __getitem__(self, idx):
        X = np.zeros((self.batch_size, self.seq_len, *self.target_size, 3))
        y = np.zeros((self.batch_size, 4))
        batch = list(self.data.values())[idx][:self.batch_size] if self.subject_aware else self.data[idx*self.batch_size:(idx+1)*self.batch_size]
        
        for i, item in enumerate(batch):
            frames, label = (item[0], item[1]) if self.subject_aware else (item[0], item[1])
            if np.random.random() > 0.5: frames = frames[::-1]
            for j, f_path in enumerate(frames):
                img = cv2.resize(cv2.cvtColor(cv2.imread(f_path), cv2.COLOR_BGR2RGB), self.target_size)
                X[i, j] = img / 255.0
            y[i, label] = 1
        return X, y

In [ ]:
import numpy as np
import tensorflow as tf

class FocalLoss(tf.keras.losses.Loss):
    def __init__(self, alpha=0.25, gamma=2.0, **kwargs):
        super().__init__(**kwargs)
        self.alpha, self.gamma = alpha, gamma
        self.cce = tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.01)
    def call(self, y_true, y_pred):
        ce = self.cce(y_true, y_pred)
        return self.alpha * (1-tf.exp(-ce))**self.gamma * ce

class CosineAnnealingWarmRestarts(tf.keras.callbacks.Callback):
    def __init__(self, T_0=15, T_mult=2, eta_min=1e-7, initial_lr=1e-5):
        super().__init__()
        self.T_0, self.T_mult, self.eta_min, self.initial_lr = T_0, T_mult, eta_min, initial_lr
        self.T_i, self.T_cur = T_0, 0
        
    def on_epoch_begin(self, epoch, logs=None):
        if self.T_cur >= self.T_i:
            self.T_cur, self.T_i = 0, self.T_i * self.T_mult
            print(f"\n[Restart] Cosine annealing restart at epoch {epoch}")
            
        new_lr = self.eta_min + (self.initial_lr - self.eta_min) * (1 + np.cos(np.pi * self.T_cur / self.T_i)) / 2
        
        # [CRITICAL FIX] Using .assign() for Keras 3 compatibility
        self.model.optimizer.learning_rate.assign(new_lr)
        self.T_cur += 1

def freeze_batch_norm_layers(model):
    for layer in model.layers:
        if isinstance(layer, layers.BatchNormalization): 
            layer.trainable = False
        elif hasattr(layer, 'layer') and isinstance(layer.layer, layers.BatchNormalization):
            layer.layer.trainable = False

# Building the Model and Generators
model = build_temporal_hd_transformer()
train_gen = ETSequenceGenerator('/kaggle/working/split_dataset/train', batch_size=16)
val_gen = ETSequenceGenerator('/kaggle/working/split_dataset/val', batch_size=16)

# --- STAGE 1 ---
print("--- STAGE 1: Settle Transformer ---")
backbone = model.layers[3] 
# Deep freeze BN inside the backbone
for l in backbone.layer.layers:
    if "batch_normalization" in l.name.lower():
        l.trainable = False
backbone.trainable = False

model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5), loss=FocalLoss(), metrics=['accuracy'])

model.fit(
    train_gen, 
    validation_data=val_gen, 
    epochs=30, 
    class_weight=class_weights,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=30, restore_best_weights=True),
        CosineAnnealingWarmRestarts(initial_lr=1e-5)
    ]
)

# --- STAGE 2 ---
print("--- STAGE 2: Deep Fine-Tuning ---")
backbone.trainable = True
for l in backbone.layer.layers:
    if "batch_normalization" in l.name.lower():
        l.trainable = True
# Protect early layers
for l in backbone.layer.layers[:-40]:
    l.trainable = False

train_gen.subject_aware = True
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-6), loss=FocalLoss(), metrics=['accuracy'])

model.fit(
    train_gen, 
    validation_data=val_gen, 
    initial_epoch=30, 
    epochs=150, 
    class_weight=class_weights,
    callbacks=[CosineAnnealingWarmRestarts(T_0=30, initial_lr=1e-6)]
)

--- STAGE 1: Settle Transformer ---
Epoch 1/30


E0000 00:00:1769252293.319379      55 meta_optimizer.cc:967] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape inStatefulPartitionedCall/functional_18_1/time_distributed_46_1/block2b_drop_1/stateless_dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer
I0000 00:00:1769252304.591289     126 cuda_dnn.cc:529] Loaded cuDNN version 91002


132/132 ━━━━━━━━━━━━━━━━━━━━ 259s 1s/step - accuracy: 0.2850 - loss: 0.2103 - val_accuracy: 0.2578 - val_loss: 0.2178
Epoch 2/30
132/132 ━━━━━━━━━━━━━━━━━━━━ 118s 889ms/step - accuracy: 0.2970 - loss: 0.1944 - val_accuracy: 0.2598 - val_loss: 0.1925
Epoch 3/30
132/132 ━━━━━━━━━━━━━━━━━━━━ 118s 888ms/step - accuracy: 0.2116 - loss: 0.2041 - val_accuracy: 0.3516 - val_loss: 0.1918
Epoch 4/30
132/132 ━━━━━━━━━━━━━━━━━━━━ 118s 887ms/step - accuracy: 0.1956 - loss: 0.2014 - val_accuracy: 0.2598 - val_loss: 0.1942
Epoch 5/30
132/132 ━━━━━━━━━━━━━━━━━━━━ 117s 885ms/step - accuracy: 0.2316 - loss: 0.1988 - val_accuracy: 0.2598 - val_loss: 0.1949
Epoch 6/30
132/132 ━━━━━━━━━━━━━━━━━━━━ 118s 887ms/step - accuracy: 0.1991 - loss: 0.2007 - val_accuracy: 0.1309 - val_loss: 0.2030
Epoch 7/30
132/132 ━━━━━━━━━━━━━━━━━━━━ 117s 885ms/step - accuracy: 0.2264 - loss: 0.1965 - val_accuracy: 0.1309 - val_loss: 0.2015
Epoch 8/30
132/132 ━━━━━━━━━━━━━━━━━━━━ 118s 887ms/step - accuracy: 0.1922 - loss: 0.2000 